# 04 — Experimento Aditivo en Colab GPU: Style-Conditioned U-Net (Style Encoder + AdaIN)

Este notebook entrena y evalúa la arquitectura **aditiva** `StyleUNetGenerator` (`src/models/style_unet.py`), desacoplando:
- **Contenido (`x`):** geometría del carácter desde `Lato-Regular` (`1×128×128`).
- **Estilo (`style_refs`):** 4 glifos de referencia del mismo autor/fuente codificados por `StyleEncoder` (sin `InstanceNorm` para preservar grosor de trazo) e inyectados mediante **AdaIN**.

Mantiene intactos tus modelos Pix2Pix anteriores (`pix2pix_pretrained.pth`, `pix2pix_scratch.pth`, `pix2pix_finetuned.pth`) y evalúa con los **mismos dos jueces congelados** (`ResNet-18` y `EfficientNet-B0`) sobre `data/real_pairs`.

**Requisitos:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

In [ ]:
# 1. Verificar GPU y montar Google Drive
from pathlib import Path
import torch
from google.colab import drive

print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/handwritten-font-cgan")
CKPT_DIR = DRIVE_ROOT / "checkpoints"
FIG_DIR = DRIVE_ROOT / "figuras"
RES_DIR = DRIVE_ROOT / "results"
for d in (CKPT_DIR, FIG_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

for req in ("judge_resnet18.pth", "judge_efficientnet_b0.pth"):
    p = CKPT_DIR / req
    print(f"  {req}: {'OK' if p.exists() else 'FALTA'}")

In [ ]:
# 2. Clonar o actualizar repositorio desde GitHub
import os
import shutil
import subprocess

REPO_URL = "https://github.com/carloamezcua/handwritten-font-cgan.git"
REPO_DIR = Path("/content/handwritten-font-cgan")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Directorio actual:", Path.cwd())
assert Path("src/train/train_style_unet.py").exists(), (
    "Falta src/train/train_style_unet.py en GitHub: haz git push origin main antes de correr este notebook."
)
print("[OK] Módulos StyleUNet verificados en el repo.")

In [ ]:
# 3. Generar dataset sintético (3,825 pares) y cargar data/real_pairs/ desde Drive
import zipfile

if not Path("data/synthetic_pairs/metadata.csv").exists():
    !python -m src.data.03_generar_sinteticos --download-fonts
else:
    print("[OK] data/synthetic_pairs ya disponible.")

LOCAL_REAL_DIR = Path("data/real_pairs")
DRIVE_ZIP = DRIVE_ROOT / "real_pairs.zip"
if DRIVE_ZIP.exists():
    if LOCAL_REAL_DIR.exists():
        shutil.rmtree(LOCAL_REAL_DIR)
    LOCAL_REAL_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        for info in zf.infolist():
            rel = info.filename.replace("\\", "/").lstrip("/")
            if not rel or rel.endswith("/"):
                continue
            if rel.startswith("real_pairs/"):
                rel = rel[len("real_pairs/"):]
            out = LOCAL_REAL_DIR / rel
            out.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info, "r") as s, out.open("wb") as d:
                shutil.copyfileobj(s, d)
    print("[OK] data/real_pairs extraído desde Drive.")
else:
    print("[AVISO] Sube real_pairs.zip a", DRIVE_ZIP)

In [ ]:
# 4. Preentrenar StyleUNet sobre las 14 fuentes sintéticas (con aumento de grosor de trazo)
STYLE_RES_DIR = str(RES_DIR / "style_unet")
!python -m src.train.train_style_unet \
    --mode pretrain \
    --data-dir data/synthetic_pairs \
    --checkpoints-dir "{CKPT_DIR}" \
    --figures-dir "{FIG_DIR}" \
    --results-dir "{STYLE_RES_DIR}" \
    --epochs 60 \
    --batch-size 16 \
    --lr 2e-4 \
    --sample-every 10 \
    --num-workers 2

In [ ]:
# 5. Fine-tuning ligero opcional sobre tus 192 trazos reales con stylus (StyleEncoder congelado)
STYLE_PRE_CKPT = str(CKPT_DIR / "style_unet_pretrained.pth")
STYLE_RES_DIR = str(RES_DIR / "style_unet")
!python -m src.train.train_style_unet \
    --mode finetune \
    --data-dir data/real_pairs \
    --pretrained-ckpt "{STYLE_PRE_CKPT}" \
    --checkpoints-dir "{CKPT_DIR}" \
    --figures-dir "{FIG_DIR}" \
    --results-dir "{STYLE_RES_DIR}" \
    --epochs 40 \
    --batch-size 16 \
    --lr 5e-5 \
    --freeze-style-encoder \
    --sample-every 10 \
    --num-workers 2

In [ ]:
# 6. Evaluación cuantitativa comparativa (Few-Shot puro vs. Style-Finetune) sobre data/real_pairs
SPEC_FEWSHOT = f"style_fewshot={CKPT_DIR / 'style_unet_pretrained.pth'}"
SPEC_FINETUNE = f"style_finetune={CKPT_DIR / 'style_unet_finetuned.pth'}"
EVAL_RES_DIR = str(RES_DIR / "eval")

!python -m eval.evaluate_style_unet \
    --data-dir data/real_pairs \
    --models "{SPEC_FEWSHOT}" "{SPEC_FINETUNE}" \
    --judges-dir "{CKPT_DIR}" \
    --results-dir "{EVAL_RES_DIR}" \
    --figures-dir "{FIG_DIR}" \
    --tag style_real

summary_path = RES_DIR / "eval" / "summary_style_real.md"
if summary_path.exists():
    print("\n" + summary_path.read_text(encoding="utf-8"))

In [ ]:
# 7. Visualizar curvas y cuadrículas (Refs estilo | Entrada x | Generados | Real y)
from IPython.display import Image, display

for fname in [
    "train_style_unet_pretrain.png",
    "train_style_unet_finetune.png",
    "style_eval_style_real_val.png",
    "style_eval_style_real_test_unseen.png",
]:
    p = FIG_DIR / fname
    if p.exists():
        print(f"\n--- {fname} ---")
        display(Image(filename=str(p)))